In [1]:
import easyocr

# Initialisation
reader = easyocr.Reader(["en"])

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.
/home/nabilath/nv_project/.venv/lib/python3.14/site-packages/torch/ao/nn/quantized/dynamic/modules/rnn.py:162: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/quantized/Quantizer.cpp:111.)
  w_ih = torch.quantize_per_tensor(


In [2]:
import os

In [3]:
path = "dataset/Resume_dataset"
files = sorted(os.listdir(path))
ocr_dict = {}
for file in files:
    img_path = os.path.join(path, file)
    results = reader.readtext(
        img_path, detail=0
    )  # detail=0 renvoie directement le texte brut
    # Affiche le texte extrait
    extracted_text = "\n".join(results)
    ocr_dict[file] = extracted_text

/home/nabilath/nv_project/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/home/nabilath/nv_project/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/home/nabilath/nv_project/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/home/nabilath/nv_project/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/home/nabilath/nv_projec

### Chunking avec Langchain

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=100, chunk_overlap=20, length_function=len
)
all_chunks = []

/home/nabilath/nv_project/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
for k, txt in ocr_dict.items():
    chunks = splitter.create_documents(
        texts=[txt],
        metadatas=[{"source": k}],
    )
    all_chunks.extend(chunks)

print(
    f"Nombre total de chunks générés pour tous les documents : {len(all_chunks)}"
)

Nombre total de chunks générés pour tous les documents : 84


In [20]:
try:
    client.delete_collection(name="rag_collection")
except ValueError:
    pass

In [16]:
import uuid

import chromadb
from chromadb.utils.embedding_functions import (
    SentenceTransformerEmbeddingFunction,
)

# chargement du modèle de création des embeddings
sentence_transformer_ef = SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2", device="cpu", normalize_embeddings=False
)

# pour éviter que tout s'efface d'ici la prochaine fois
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(
    name="rag_collection",
    embedding_function=sentence_transformer_ef,  # on lie le modèle à chromadb
)

# ajout des chunks dans la collection
collection.add(
    documents=[chunk.page_content for chunk in all_chunks],
    metadatas=[chunk.metadata for chunk in all_chunks],
    ids=[str(uuid.uuid4()) for _ in all_chunks],
)

In [17]:
# test d'une requête et récupération des chunks les plus proches
results = collection.query(query_texts=["Qui a un PhD ?"], n_results=5)
results["documents"][0]

['1986\nMD)_\nUniversity of Rouen, France\n1990 PhD\nUniversity 0f Paris\nFrancc\nPredoctoral Training:',
 'CURRICULUM VITAF\nName:\nThierry Frebourg\nAddress:\n22 rue des\nBoulangers, 75005 Paris , France',
 'California, Berkeley,\nCalifornia.\n1966\n1971\nAssociate\nProfessor\nin Residence, Departnent',
 'University\nof California\nBerkeley ,\nCalifornia.\n1958\nNational\nScience\nFoundation Predoctoral Fellow',
 'Date of Pirth:\nJune 18,1990\nPlace of Birth:\nDieppe , France\nFducation:\n1986\nMD)_']

## Pipeline RAG

In [8]:
def retrieval(collection, query, top_k=5):
    results = collection.query(query_texts=[query], n_results=top_k)
    documents = results["documents"][0]
    metadatas = results["metadatas"][0]

    res_list = []
    for doc, meta in zip(documents, metadatas):
        res_list.append({"content": doc, "metadata": meta})

    return res_list

In [18]:
retrieval(collection, "Qui a un PhD ?")

[{'content': '1986\nMD)_\nUniversity of Rouen, France\n1990 PhD\nUniversity 0f Paris\nFrancc\nPredoctoral Training:',
  'metadata': {'source': '40028776-8777.jpg'}},
 {'content': 'CURRICULUM VITAF\nName:\nThierry Frebourg\nAddress:\n22 rue des\nBoulangers, 75005 Paris , France',
  'metadata': {'source': '40028776-8777.jpg'}},
 {'content': 'California, Berkeley,\nCalifornia.\n1966\n1971\nAssociate\nProfessor\nin Residence, Departnent',
  'metadata': {'source': '50267812-7820.jpg'}},
 {'content': 'University\nof California\nBerkeley ,\nCalifornia.\n1958\nNational\nScience\nFoundation Predoctoral Fellow',
  'metadata': {'source': '50267812-7820.jpg'}},
 {'content': 'Date of Pirth:\nJune 18,1990\nPlace of Birth:\nDieppe , France\nFducation:\n1986\nMD)_',
  'metadata': {'source': '40028776-8777.jpg'}}]

In [10]:
def build_prompt(context, question):
    return f"""
            You are a helpful assistant to give informations about resumes provided.
            You should respond in the language of the question. You should always
            be polite.
            If you don't know the answer or if it's not present in the context,
            you should state that you do not have the information.
            Context:
            {context}
            Question:
            {question}
            """

In [ ]:
import ollama


def ask_rag(collection, question, top_k=5):
    # RETRIEVAL : récupérer les chunks pertinents depuis ChromaDB
    results = collection.query(query_texts=[question], n_results=top_k)

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]

    # Concaténer les chunks en un seul bloc de contexte propre
    context_blocks = []
    for doc, meta in zip(documents, metadatas):
        source_name = meta.get("source", "Inconnu")
        context_blocks.append(f"--- Source: {source_name} ---\n{doc}")

    context = "\n\n".join(context_blocks)

    # PROMPT : Construire le prompt avec ta fonction précédente
    prompt = build_prompt(context, question)

    # 3. LLM : Envoyer le tout à Llama 3.2 via Ollama
    response = ollama.chat(
        model="llama3.2", messages=[{"role": "user", "content": prompt}]
    )

    # Retourner la réponse du modèle et les sources pour vérification
    return {
        "answer": response["message"]["content"],
        "sources": metadatas,
        "context_used": context,
    }

In [57]:
ask_rag(collection, "Qui a un PhD ?")

{'answer': "Une question simple, mais intéressante !\n\nSelon les informations fournies, c'est Thierry Frebourg qui a un PhD (Doctorat en Philosophie). Il a obtenu ce titre en 1990 à l'Université de Paris.",
 'sources': [{'source': '40028776-8777.jpg'},
  {'source': '40028776-8777.jpg'},
  {'source': '50267812-7820.jpg'},
  {'source': '50267812-7820.jpg'},
  {'source': '40028776-8777.jpg'}],
 'context_used': '--- Source: 40028776-8777.jpg (Page Inconnue) ---\n1986\nMD)_\nUniversity of Rouen, France\n1990 PhD\nUniversity 0f Paris\nFrancc\nPredoctoral Training:\n\n--- Source: 40028776-8777.jpg (Page Inconnue) ---\nCURRICULUM VITAF\nName:\nThierry Frebourg\nAddress:\n22 rue des\nBoulangers, 75005 Paris , France\n\n--- Source: 50267812-7820.jpg (Page Inconnue) ---\nCalifornia, Berkeley,\nCalifornia.\n1966\n1971\nAssociate\nProfessor\nin Residence, Departnent\n\n--- Source: 50267812-7820.jpg (Page Inconnue) ---\nUniversity\nof California\nBerkeley ,\nCalifornia.\n1958\nNational\nScience\nFo

In [11]:
from sentence_transformers import CrossEncoder

encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2854.78it/s]


In [ ]:
def retrieval_with_reranker(collection, encoder, query, top_k=5):
    results = collection.query(query_texts=[query], n_results=top_k)
    documents = results["documents"][0]
    metadatas = results["metadatas"][0]

    pairs = [(query, doc) for doc in documents]

    scores = encoder.predict(pairs)
    combined = list(zip(scores, documents, metadatas))
    reranked = sorted(combined, key=lambda x: x[0], reverse=True)

    res_list = []
    for score, doc, meta in reranked:
        res_list.append(
            {"content": doc, "metadata": meta, "score": float(score)}
        )

    return res_list

In [19]:
retrieval_with_reranker(collection, encoder, "Qui a un PhD ?")

[{'content': '1986\nMD)_\nUniversity of Rouen, France\n1990 PhD\nUniversity 0f Paris\nFrancc\nPredoctoral Training:',
  'metadata': {'source': '40028776-8777.jpg'},
  'score': -10.216346740722656},
 {'content': 'University\nof California\nBerkeley ,\nCalifornia.\n1958\nNational\nScience\nFoundation Predoctoral Fellow',
  'metadata': {'source': '50267812-7820.jpg'},
  'score': -10.903210639953613},
 {'content': 'California, Berkeley,\nCalifornia.\n1966\n1971\nAssociate\nProfessor\nin Residence, Departnent',
  'metadata': {'source': '50267812-7820.jpg'},
  'score': -11.10226821899414},
 {'content': 'CURRICULUM VITAF\nName:\nThierry Frebourg\nAddress:\n22 rue des\nBoulangers, 75005 Paris , France',
  'metadata': {'source': '40028776-8777.jpg'},
  'score': -11.385540008544922},
 {'content': 'Date of Pirth:\nJune 18,1990\nPlace of Birth:\nDieppe , France\nFducation:\n1986\nMD)_',
  'metadata': {'source': '40028776-8777.jpg'},
  'score': -11.485108375549316}]